## Notebook 3 — Models

## 1. Imports and Load Data

load the four processed CSV files saved by the preprocessing notebook

`joblib` is used to save trained models

`.squeeze()` is applied to y files because pandas loads a single-column CSV as a DataFrame

In [1]:
import pandas as pd # data loading and manipulation
import os # builds file paths for Mac and Windows
import joblib # saving and loading models
from sklearn.linear_model import LogisticRegression # baseline model
from sklearn.ensemble import RandomForestClassifier # primary model
from sklearn.metrics import classification_report # precision, recall, F1 in one table

# Load the four processed CSV files
# X = features (30 columns: V1-V28, Log_Amount, Hour)
# y = target (0 = legitimate, 1 = fraud)
X_train = pd.read_csv(os.path.join('..', 'data', 'processed', 'X_train.csv'))
y_train = pd.read_csv(os.path.join('..', 'data', 'processed', 'y_train.csv')).squeeze()

X_test = pd.read_csv(os.path.join('..', 'data', 'processed', 'X_test.csv'))
y_test = pd.read_csv(os.path.join('..', 'data', 'processed', 'y_test.csv')).squeeze()

print('Data loaded successfully')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape},  y_test:  {y_test.shape}')

Data loaded successfully
X_train: (227845, 30), y_train: (227845,)
X_test:  (56962, 30),  y_test:  (56962,)


## 2. Logistic Regression — Baseline Model

Logistic Regression is trained first as a **baseline** it gives us a reference point
to measure how much better Random Forest performs

It works by calculating a probability between 0 and 1 for each transaction

**Why `max_iter=1000`?**
improves its predictions step by step
default 100 steps is not enough for a dataset this large
Setting max_iter=1000 gives it enough steps to fully converge

**Key weakness:**
Fraud patterns are complex and non-linear

In [2]:
# Create Logistic Regression model
lr = LogisticRegression(max_iter=1000)

# fit() trains the model — it learns the weights for each feature from training data
lr.fit(X_train, y_train)

predict_y_train = lr.predict(X_train) # predictions on training data
predict_y_test = lr.predict(X_test) # predictions on test data

# classification_report prints Precision, Recall, F1 and Support for each class
print(classification_report(y_test, predict_y_test, target_names=['Legitimate', 'Fraud']))

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56864
       Fraud       0.83      0.64      0.72        98

    accuracy                           1.00     56962
   macro avg       0.91      0.82      0.86     56962
weighted avg       1.00      1.00      1.00     56962



## 3. Random Forest — Primary Model

Random Forest builds **multiple decision trees** and combines their predictions through majority vote/Each tree asks a series of yes/no questions about the features

**Gini index** determines which question to ask at each split

**Why `n_estimators=100`?**
This sets the number of trees (100 trees as stable predictions without excessive training time)







In [3]:
# Create Random Forest model
# n_estimators=100 — builds 100 decision trees
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# fit() trains all 100 trees on the training data
rf.fit(X_train, y_train)

predict_rf_train = rf.predict(X_train) # predictions on training data
predict_rf_test = rf.predict(X_test) # predictions on test data

print(classification_report(y_test, predict_rf_test, target_names=['Legitimate', 'Fraud']))

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56864
       Fraud       0.94      0.81      0.87        98

    accuracy                           1.00     56962
   macro avg       0.97      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962



## 4. Save Models

Both trained models are saved to the `models/` folder using `joblib`

**Why save the models?**
Saving the trained model means Notebook 4 can load it instantly

**What is `.pkl`?**
Pickle format a way of saving any Python object to a file

In [4]:
# Create the models folder if it does not already exist
os.makedirs(os.path.join('..', 'models'), exist_ok=True)

# Save both trained models
joblib.dump(lr, os.path.join('..', 'models', 'logistic_regression.pkl'))
joblib.dump(rf, os.path.join('..', 'models', 'random_forest.pkl'))

print('Models saved to models/')

Models saved to models/
